# Mobilidade Urbana no Brasil — Notebook de Análise
**Disciplina:** Linguagens de Programação — Análise e Visualização de Dados com Python (G1)
**Aluna:** Juliana Neto Sá | **Professor:** Alexandre Neves Louzada

## 1. Introdução ao problema
O deslocamento diário nas cidades brasileiras envolve tempo perdido, custo de tarifa, lotação e emissões de CO₂. Quais meios de transporte, regiões e períodos apresentam os piores indicadores? Como as variáveis se relacionam?

## 2. Explicação da base de dados
`simulacao_mobilidade_urbana_brasil.csv` (AlexandreLouzada/Dados-Simulados-G2) — dados **simulados**, uso didático. Cada linha é um registro mensal de uma cidade e um meio de transporte.

| Coluna | Descrição |
|---|---|
| ano, mes, data | Período do registro (2015–2024) |
| regiao, uf, cidade | Localização (5 regiões, 20 UFs, 37 cidades) |
| meio_transporte | Ônibus, BRT, Metrô, Trem, VLT, Bicicleta |
| passageiros | Passageiros transportados no mês |
| tempo_medio_deslocamento | Minutos |
| lotacao_media | Lotação (%) |
| velocidade_media | km/h |
| emissao_co2 | Emissão de CO₂ (unidade da base) |
| tarifa_media | R$ |
| nivel_congestionamento | Baixo, Médio, Alto, Crítico |

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
URL = "https://raw.githubusercontent.com/AlexandreLouzada/Dados-Simulados-G2/main/datasets_g2_30_temas/simulacao_mobilidade_urbana_brasil.csv"

## 3. Leitura dos dados
O arquivo começa com BOM (caractere invisível), por isso usamos `encoding='utf-8-sig'`.

In [ ]:
try:
    df = pd.read_csv("../dados/simulacao_mobilidade_urbana_brasil.csv", encoding="utf-8-sig")
except FileNotFoundError:
    df = pd.read_csv(URL, encoding="utf-8-sig")
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

## 4. Limpeza e preparação

In [ ]:
df.columns = [c.replace("\ufeff", "").strip() for c in df.columns]
print("Nulos:", df.isna().sum().sum(), "| Duplicados:", df.duplicated().sum())
df = df.drop_duplicates()
df["data"] = pd.to_datetime(df["data"])
for c in ["regiao", "uf", "cidade", "meio_transporte", "nivel_congestionamento"]:
    df[c] = df[c].str.strip()
ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]
df["nivel_congestionamento"] = pd.Categorical(df["nivel_congestionamento"], categories=ORDEM, ordered=True)
df.dtypes

## 5. Engenharia de atributos

In [ ]:
df["indice_congestionamento"] = df["nivel_congestionamento"].cat.codes + 1      # 1=Baixo ... 4=Crítico
df["congestionado"] = df["indice_congestionamento"] >= 3                        # Alto ou Crítico
df["emissao_por_mil_passageiros"] = df["emissao_co2"] / df["passageiros"] * 1000
df["trimestre"] = df["data"].dt.quarter
df[["indice_congestionamento", "congestionado", "emissao_por_mil_passageiros", "trimestre"]].head()

## 6. Análise exploratória

In [ ]:
nums = ["passageiros", "tempo_medio_deslocamento", "lotacao_media", "velocidade_media", "emissao_co2", "tarifa_media"]
df[nums].hist(figsize=(12, 7), bins=30, color="#2a9d8f"); plt.tight_layout(); plt.show()

In [ ]:
print(df["nivel_congestionamento"].value_counts(normalize=True).round(3))
df.groupby("meio_transporte")[nums].mean().round(1)

In [ ]:
df.groupby('regiao')[nums].mean().round(1)

## 7. KPIs

In [ ]:
kpis = {
    "Registros": len(df),
    "Passageiros (milhões)": df["passageiros"].sum() / 1e6,
    "Tempo médio (min)": df["tempo_medio_deslocamento"].mean(),
    "Velocidade média (km/h)": df["velocidade_media"].mean(),
    "Tarifa média (R$)": df["tarifa_media"].mean(),
    "% registros Alto/Crítico": df["congestionado"].mean() * 100,
}
pd.Series(kpis).round(2)

## 8. Gráficos

In [ ]:
g = df.groupby("meio_transporte")["tempo_medio_deslocamento"].mean().sort_values(ascending=False)
plt.figure(figsize=(7, 4)); sns.barplot(x=g.values, y=g.index, color="#2a9d8f")
plt.title("Tempo médio de deslocamento por meio de transporte (min)"); plt.xlabel("min"); plt.ylabel(""); plt.show()

In [ ]:
plt.figure(figsize=(9, 4.5)); sns.boxplot(data=df, x="regiao", y="tempo_medio_deslocamento", color="#8ecae6")
plt.title("Tempo de deslocamento por região"); plt.show()

In [ ]:
s = df.assign(p=df["data"].dt.to_period("M").dt.to_timestamp()).groupby("p")["passageiros"].mean()
plt.figure(figsize=(11, 4)); plt.plot(s.index, s.values, alpha=.5, label="Mensal")
plt.plot(s.index, s.rolling(12, min_periods=3).mean(), lw=2.5, label="Média móvel 12m")
plt.title("Passageiros médios por registro ao longo do tempo"); plt.legend(); plt.show()

In [ ]:
ct = pd.crosstab(df["meio_transporte"], df["nivel_congestionamento"], normalize="index") * 100
ct.plot(kind="barh", stacked=True, figsize=(10, 4), color=["#90be6d", "#f9c74f", "#f8961e", "#d62828"])
plt.xlabel("% dos registros"); plt.title("Congestionamento por meio de transporte"); plt.show()

In [ ]:
cols = nums + ["indice_congestionamento", "emissao_por_mil_passageiros"]
plt.figure(figsize=(9, 6.5)); sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlação de Pearson"); plt.show()

## 9. Interpretação dos resultados
Valores obtidos ao executar o notebook com a base completa (revise com seus próprios gráficos):

- **Tempo de deslocamento:** média geral de ~54,3 min. Entre os meios, a Bicicleta tem a menor média (~52,7 min) e o BRT a maior (~55,7 min); entre as regiões, o Centro-Oeste tem a menor (~52,3) e o Sul a maior (~55,4). As diferenças são de poucos minutos e as distribuições se sobrepõem bastante.
- **Passageiros:** o Metrô tem a maior média por registro (~1,04 milhão); por ano, 2019 foi o pico e 2022 o menor valor da série.
- **Congestionamento:** os quatro níveis são quase equilibrados (Médio 26,5%, Baixo 25,2%, Alto 24,8%, Crítico 23,5%); cerca de 48% dos registros estão em nível Alto ou Crítico.
- **Correlação:** todas as correlações entre as variáveis numéricas ficam próximas de zero (|r| ≤ 0,03), ou seja, não há relação linear relevante, por exemplo entre velocidade e tempo de deslocamento.
- **Observação crítica:** esse conjunto de resultados é típico de dados simulados com valores sorteados de forma independente. Em dados reais esperaríamos, por exemplo, correlação negativa entre velocidade e tempo de deslocamento.

## 10. Conclusão
A base não revela diferenças fortes entre meios de transporte, regiões ou anos, nem relações lineares entre as variáveis, o que é coerente com a natureza simulada dos dados. O projeto cumpre o objetivo técnico (limpeza, KPIs, visualização e dashboard interativo), e a conclusão analítica é que, **com estes dados, não há evidência para priorizar um meio ou região específicos**; seria necessário validar com dados reais (e variáveis como população, renda e infraestrutura) antes de recomendar ações.